# Workshop CFE — Data Engineering en Databricks

Bienvenidos. Durante las sesiones construiremos, de punta a punta, un pipeline de
**analytics de proyectos de construcción** — del Parquet crudo a una capa **oro** lista
para dashboards y análisis.

## Datos del workshop
La ruta de datos se configura en `config.py` (variable `DATA_PATH`).
Son **54 archivos Parquet** exportados desde un backup de SQL Server (`UniversoProyectos_230726.bak`)
que contiene el universo de proyectos de construcción de CFE: obras, proyectos, suministros,
bitácoras, catálogos de estados, tipos de obra, etc.

Tablas principales:
* **`tbobras`** (318 filas, 50 cols) — obras de construcción con avance, importes, fechas
* **`tbproyectos`** (78 filas, 72 cols) — proyectos con alcance, estado, coordenadas, montos
* **`ctestadosrepublica`** — catálogo de estados de la república
* **`ctestadosproyectos`** — catálogo de estados de proyecto
* **`cttipoobra`** — tipos de obra (línea, subestación)

## Cómo trabajar
1. Edita **`config.py`**: `CATALOG`, `WORK_SCHEMA`, `DATA_PATH`.
2. Cada notebook empieza con `%run ./_setup` que lee config, crea tu esquema y expone variables.
3. Ejecuta celda por celda.

## Agenda
| # | Notebook | Tema |
|---|---|---|
| 01 | `01_delta_lab` | Tablas Delta: crear desde Parquet, MERGE, time travel |
| 02 | `02_autoloader_lab` | Auto Loader: ingesta incremental, checkpoint, `_metadata`, schema evolution |
| 03 | `03_declarative_pipeline` | Lakeflow Declarative Pipelines: bronze(54 tablas)→silver→gold (modelo estrella) |
| 04 | `04_semantic_layer` | Capa semántica: Metric Views sobre el modelo estrella |
| 05 | `05_dashboard_ejecutivo` | AI/BI Dashboards: dashboard ejecutivo desde las metric views del lab 04 |
| 06 | `06_app_notas_obra_lakebase` | Databricks Apps + Lakebase: app de notas de campo desplegada desde la interfaz |
| 07 | `07_unity_catalog_lab` | Unity Catalog: permisos, linaje, tags, masking |
| 08 | `08_optimizacion_delta_lab` | Liquid Clustering, ZSTD, file sizing |
| 09 | `09_modelo_riesgo_obras` | Modelo de riesgo: scikit-learn + MLflow + Unity Catalog Model Registry |
| 10 | `10_reentrenamiento_modelo` | Reentrenamiento automático semanal con drift detection |

## Prerrequisitos del instructor (antes del workshop)

### 1. Permisos sobre system tables (lab 07)
El lab 07 incluye una actividad donde los participantes crean un dashboard de monitoreo
con **system tables**. Un administrador debe otorgar acceso antes de la sesión:

```sql
-- Ajusta el nombre del grupo según tu workspace
GRANT USE SCHEMA ON SCHEMA system.query  TO `participantes_workshop`;
GRANT SELECT     ON SCHEMA system.query  TO `participantes_workshop`;
GRANT USE SCHEMA ON SCHEMA system.access TO `participantes_workshop`;
GRANT SELECT     ON SCHEMA system.access TO `participantes_workshop`;
```

Tablas utilizadas: `system.query.history`, `system.access.audit`,
`system.access.table_lineage`, `system.access.column_lineage`,
`system.access.assistant_events`.

### 2. Pipeline del lab 03
El pipeline medallion (`pipeline_cfe_medallion`) debe haber corrido al menos una vez
para que existan las tablas `gold_*`, `silver_*` y `bronze_*` que usan los labs 04-10.

### 3. Lakebase (lab 06)
Cada participante crea su propio proyecto de Lakebase y una synced table.
Verifica que el workspace tenga habilitada la funcionalidad de Lakebase Postgres.

In [0]:
%run ./_setup